# Example 2: speckle field metrics on a speckle scan

**Goal.** Load an experimental speckle stack, apply flat-field correction, compute frame-wise speckle metrics, select a representative frame, and inspect its spatial speckle distribution.

This example shows the two complementary levels of the API:

- `dip.speckle_stack_stats(...)` for **stack analysis**
- `dip.speckle_stats(...)` for **single-image analysis**

In [ ]:
__author__ = ['Rafael Celestre']
__contact__ = 'rafael.celestre@esrf.eu'
__license__ = 'CECILL-2.1'
__copyright__ = 'ESRF - The European Synchrotron'
__created__ = '2026.03.06'
__changed__ = '2026.03.24'

import logging
import sys

import barc4dip as dip
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(message)s")

print(sys.executable)
print(sys.version)

%load_ext autoreload
%autoreload 2

# %matplotlib widget
# %matplotlib inline

## 1) Load a speckle scan

Replace `data_path` with the location of the example dataset.  
Later this can point to the reproducible dataset archived in Zenodo.

In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/speckle_tracking/speckles.h5"

scan = dip.read_image(data_path, verbose=True)
scan.shape

In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/speckle_tracking/darks.h5"

darks = dip.read_image(data_path, mean=True, verbose=True)
darks.shape

In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/speckle_tracking/flats.h5"

flats = dip.read_image(data_path, mean=True, verbose=True)
flats.shape

## 2) Apply flat-field correction

For experimental speckle data, correcting detector offsets and illumination non-uniformity is usually worth doing before computing metrics.


In [ ]:
scan = dip.preprocessing.flat_field_correction(scan, flats=flats, darks=darks, verbose=True)

## 3) Quick visual check

A fast image + histogram view is useful before computing metrics.

In [ ]:
def show_image_summary(image: np.ndarray, *, title: str | None = None,
                       bin_min: int = 0, bin_max: int = 65535, hist: bool = True, ) -> None:
    
    pmin, pmax = 1., 99.

    vmin = np.nanpercentile(image, pmin)
    vmax = np.nanpercentile(image, pmax)

    dip.plotting.plt_image(
        image,
        title=title,
        vmin=vmin,
        vmax=vmax,
        cmap="srw",
        display_origin="lower",
    )

    if hist:
        dip.plotting.plt_histogram(
            image,
            logy=True,
            cumulative=True,
            percentiles=(1, 99),
            bin_min=bin_min,
            bin_max=bin_max,
        )

    dip.plotting.show()

show_image_summary(scan[0], bin_min=100, bin_max=25000, title="first frame (flat field corrected)")

## 4) Compute stack speckle metrics

The stack function evaluates each frame and, optionally, image tiles.  
In this example we deliberately use `scope="both"` in the plots, so the full-frame trace and the tile statistics are displayed together in a single call.


In [ ]:
stack_stats = dip.speckle_stack_stats(scan)

## 5) Compare a few representative speckle indicators

We show three metrics that capture complementary aspects of the speckle field:

- `amplitude.visibility`: intensity fluctuation strength
- `grain.lx`: horizontal speckle grain size from the autocorrelation peak
- `grain.ly`: vertical speckle grain size from the autocorrelation peak


In [ ]:
dip.plotting.plt_stack_metric(stack_stats, "amplitude.visibility", scope="both", uncertainty="band")
dip.plotting.show()

In [ ]:
dip.plotting.plt_stack_metric(stack_stats, "grain.lx", scope="both", uncertainty="band")
dip.plotting.show()

In [ ]:
dip.plotting.plt_stack_metric(stack_stats, "grain.ly", scope="both", uncertainty="band")
dip.plotting.show()

## 6) Select lowest visibility frame

Here we use **Tenengrad** as the focus score.

In [ ]:
best_focus_frame = int(np.argmin(stack_stats["full"]["amplitude"]["visibility"]))
print(f">>> lowest visibility at frame #{best_focus_frame}")

frame = scan[best_focus_frame]
vmin = np.nanpercentile(frame, 1.0)
vmax = np.nanpercentile(frame, 99.0)

show_image_summary(frame, title=f"Lowest-visibility frame #{best_focus_frame}", bin_max=25000)

## 7) Compute single-image speckle statistics

Once the best frame is identified, we can inspect all sharpness blocks in detail.

In [ ]:
frame_stats = dip.speckle_stats(frame, verbose=True)
frame_stats

## 8 Spatial distribution across tiles

This plot answers a practical question: **is the image sharp everywhere, or only locally?**

In [ ]:
dip.plotting.plt_tiles_metric(
    frame,
    frame_stats,
    "amplitude.visibility",
    show_std=True,
    vmin=vmin,
    vmax=vmax,
    fmt="{:.3f}"
)
dip.plotting.show()

In [ ]:
dip.plotting.plt_tiles_metric(
    frame,
    frame_stats,
    "grain.lx",
    show_std=True,
    vmin=vmin,
    vmax=vmax,
    fmt="{:.1f}"
)
dip.plotting.show()

In [ ]:
dip.plotting.plt_tiles_metric(
    frame,
    frame_stats,
    "grain.ly",
    show_std=True,
    vmin=vmin,
    vmax=vmax,
    fmt="{:.1f}"
)
dip.plotting.show()

## 9) Individual metrics

A few representative metrics can also be called directly.

- **Amplitude**: visibility and robust Michelson contrast
- **Grain**: correlation-based speckle size and anisotropy
- **Bandwidth**: PSD-based equivalent bandwidth and spectral participation ratio
- **Distribution moments**: mean, standard deviation, skewness, kurtosis, and saturation statistics

In [ ]:
%timeit dip.metrics.statistics.distribution_moments(frame)
%timeit dip.metrics.speckles.grain(frame)
%timeit dip.metrics.speckles.amplitude(frame)
%timeit dip.metrics.speckles.bandwidth(frame)

In [ ]:
moments = dip.metrics.statistics.distribution_moments(frame, verbose=True)
grain = dip.metrics.speckles.grain(frame, verbose=True)
amplitude = dip.metrics.speckles.amplitude(frame, verbose=True)
bandwidth = dip.metrics.speckles.bandwidth(frame, verbose=True)

## 10) Generate a compact report

This is convenient for logbooks, beamtime notes, or regression checks.

In [ ]:
print(dip.logbook_report(frame_stats, 'speckle_image.md', complete=True, notes=True))

### Reporting via the terminal

In [ ]:
! barc4dip-speckles -h